# EDA & Feature Engineering — Walmart Store Sales Forecasting

**Shared notebook** (done together by the team). Here we understand the data
and design the features that every model reuses through `src/pipeline.py`.

**Task.** Weekly sales per `Store` x `Dept`. Metric = **WMAE** (holiday weeks
weighted 5x). Files: `train.csv`, `test.csv`, `features.csv`, `stores.csv`.

Download the data from the competition page and unzip into `./data/` first.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src.data import load_raw
from src.pipeline import build_preprocessor, RAW_COLS

raw = load_raw("data")
train, test, features, stores = raw.train, raw.test, raw.features, raw.stores
print("train", train.shape, "| test", test.shape,
      "| features", features.shape, "| stores", stores.shape)
train.head()

## 1. Target: `Weekly_Sales`
Note the negative values (markdown returns) and the heavy right skew.

In [ ]:
print(train["Weekly_Sales"].describe())
print("\nnegative sales rows:", int((train["Weekly_Sales"] < 0).sum()))
print("date range:", train["Date"].min().date(), "->", train["Date"].max().date())
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
train["Weekly_Sales"].clip(-5000, 60000).hist(bins=80, ax=ax[0]); ax[0].set_title("Weekly_Sales")
np.log1p(train["Weekly_Sales"].clip(lower=0)).hist(bins=80, ax=ax[1]); ax[1].set_title("log1p(sales)")
plt.tight_layout(); plt.show()

## 2. Seasonality — total sales over time
Clear yearly seasonality with sharp Thanksgiving / Christmas peaks.

In [ ]:
weekly = train.groupby("Date")["Weekly_Sales"].sum()
plt.figure(figsize=(12, 3))
plt.plot(weekly.index, weekly.values)
for d in train.loc[train["IsHoliday"], "Date"].unique():
    plt.axvline(pd.Timestamp(d), color="orange", alpha=0.25)
plt.title("Total weekly sales (orange = holiday weeks)"); plt.tight_layout(); plt.show()

## 3. Store structure — Type & Size

In [ ]:
print(stores["Type"].value_counts())
merged = train.merge(stores, on="Store", how="left")
print(merged.groupby("Type")["Weekly_Sales"].mean())
merged.groupby("Type")["Weekly_Sales"].mean().plot(kind="bar", figsize=(5, 3),
    title="Mean weekly sales by store type"); plt.tight_layout(); plt.show()

## 4. Holiday effect & missing values

In [ ]:
print("mean sales holiday vs non-holiday:")
print(train.groupby("IsHoliday")["Weekly_Sales"].mean(), "\n")
print("missing-value rate in features.csv:")
print(features.isna().mean().sort_values(ascending=False).round(3))

## 5. Feature engineering output
The `build_preprocessor` chain (merge -> calendar/holiday features -> impute ->
finalize) turns the 4 raw columns into the full numeric feature matrix every
model consumes. Because it lives in a Pipeline, the **same** transform runs on
the raw test set at inference time.

In [ ]:
pre = build_preprocessor(features, stores)
Xt = pre.fit_transform(train[RAW_COLS], train["Weekly_Sales"])
print("engineered features:", Xt.shape[1])
print(list(Xt.columns))
Xt.head()

## Takeaways
- Strong yearly seasonality + holiday spikes -> calendar & named-holiday flags matter.
- Store `Type`/`Size` separate sales levels -> keep as features.
- MarkDowns are missing before Nov-2011 -> fill with 0; other numerics -> median.
- Negative sales exist -> keep raw target; clip predictions at 0 for submission.
- Validation must be **time-based** (see `src/validation.py`).